# Classificação de Estrelas

Pré-processamento dos dados, definição do modelo, treino e avaliação de desempenho.



In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

df = pd.read_csv('star_classification.csv') 
df = df.dropna()

y = df['class'].values
to_rem = ['obj_ID', 'run_ID', 'rerun_ID', 'cam_col', 'field_ID', 'spec_obj_ID', 'plate', 'MJD', 'fiber_ID', 'class']
X = df.drop(columns = to_rem).select_dtypes(include = [np.number]).values

# ('GALAXY', 'STAR', 'QSO') -> (0, 1, 2)
le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

class Model(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout_p= 0.0):
        super(Model, self).__init__()
        layers = []
        
        # input layer
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())
        if dropout_p > 0: 
            layers.append(nn.Dropout(dropout_p))
        # hidden layer
        for i in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout_p > 0: 
                layers.append(nn.Dropout(dropout_p))
        # output layer
        layers.append(nn.Linear(hidden_dim, 3))

        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)


def solve(hidden_dim, num_layers, lr, epochs, dropout = 0.0, weight_decay = 0.0):
    model = Model(X_train.shape[1], hidden_dim, num_layers, dropout)
    crit = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr = lr, weight_decay = weight_decay)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = crit(outputs, y_train)
        loss.backward() # backpropagation
        optimizer.step()         
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test)
        _, predicted = torch.max(test_outputs, 1)
        correct = (predicted == y_test).sum().item()
        accur = correct / y_test.size(0)
        
    print(f"Camadas: {num_layers} | Largura: {hidden_dim} | LR: {lr} | Qtd Épocas: {epochs} | Loss Final: {loss.item():.4f} | Acc Teste: {accur * 100:.2f}%")
    return accur



print("1.b TESTANDO LARGURA E PROFUNDIDADE")
solve(hidden_dim = 32, num_layers = 1, lr = 0.001, epochs = 30)
solve(hidden_dim = 128, num_layers = 1, lr = 0.001, epochs = 30)
solve(hidden_dim = 32, num_layers = 4, lr = 0.001, epochs = 30)

print("\n1.c IDENTIFICANDO UNDERFITTING E OVERFITTING")
# pouquíssimas épocas e LR extremamente baixo
solve(hidden_dim = 64, num_layers = 2, lr = 0.00001, epochs = 3)
# muitas épocas, rede complexa, sem regularização
solve(hidden_dim = 256, num_layers = 4, lr = 0.01, epochs = 150)

print("\n1.d TESTANDO REGULARIZAÇÃO E OTIMIZAÇÃO")
# mesma configuração do overfitting, mas aplicando dropout e weigth decay 
solve(hidden_dim = 256, num_layers = 4, lr = 0.001, epochs = 100, dropout = 0.3, weight_decay = 1e-4)

### 1.b Testando Largura e Profundidade

Camadas: 1 | Largura: 32 | LR: 0.001 | Épocas: 30 | Loss Final: 0.9589 | Acc Teste: 70.87%

Camadas: 1 | Largura: 128 | LR: 0.001 | Épocas: 30 | Loss Final: 0.7883 | Acc Teste: 67.69%

Camadas: 4 | Largura: 32 | LR: 0.001 | Épocas: 30 | Loss Final: 0.9603 | Acc Teste: 59.30%

**Análise:** Aumentar a largura para 128 não melhorou acurácia neste caso; modelos mais profundos (4 camadas) com mesma largura apresentaram pior desempenho, possivelmente por overfitting ou dificuldade de otimização nas configurações usadas.

### 1.c Identificando Underfitting e Overfitting

Camadas: 2 | Largura: 64 | LR: 1e-05 | Épocas: 3 | Loss Final: 1.1293 | Acc Teste: 21.75%

Camadas: 4 | Largura: 256 | LR: 0.01 | Épocas: 150 | Loss Final: 0.1089 | Acc Teste: 96.30%

**Análise:** A primeira configuração (LR muito baixa e poucas épocas) indica underfitting, perda alta e baixa acurácia. A segunda mostra overfitting controlado ou alto poder de generalização (perda muito baixa e acurácia alta), possivelmente por combinação de alta capacidade e muitas épocas.

### 1.d TESTANDO REGULARIZAÇÃO E OTIMIZAÇÃO

Camadas: 4 | Largura: 256 | LR: 0.001 | Épocas: 100 | Loss Final: 0.2132 | Acc Teste: 94.67%

**Análise:** Aplicando `dropout=0.3` e `weight_decay=1e-4` manteve-se alta acurácia com perda moderada, indica que a regularização ajudou a reduzir overfitting mantendo bom desempenho.